# Knee v6: ConvNeXt screen on fold 0 (bigger backbone + learning-rate check), with a stall watchdog
GPU 0: ConvNeXt-Small (`convnext_small.in12k_ft_in1k`, lr_backbone 5e-5)  ~3 h.
GPU 1, one after the other: ConvNeXt-T at lr_backbone 2.5e-5, then at 1e-4  ~1.6 h each. The DINO screens showed learning rate moves fold-0 AUC by 0.1, and ConvNeXt-T has only ever run at 5e-5.
Reference (ConvNeXt-T, lr 5e-5, seed 42): fold-0 val macro (raw) ep1 0.700 | ep3 0.763 | ep5 0.800 | ep9 0.828 | ep12 0.832. Seed 43: 0.832. Seed noise ~0.01: only act on a gain above ~+0.015.
WATCHDOG: an arm is killed if it prints no epoch line within 30 min, or stalls for 45 min afterwards (a stalled geometry run burned 8 h of quota). Exit code -9 means the OS killed it, usually out of RAM.
Inputs: competition data, `knee-mri-pipeline` (patched knee_pipeline_v6), caching output, training output (oof_all.csv), the ConvNeXt 5-fold output. GPU T4 x2, internet ON, Save & Run All. ~3.2 h wall.

In [1]:
import os, sys, json, subprocess, time, threading
from pathlib import Path
import numpy as np, pandas as pd

CNX_T = "convnext_tiny.in12k_ft_in1k"
CNX_S = "convnext_small.in12k_ft_in1k"
W = Path(os.environ.get("KNEE_WORK", "/kaggle/working"))
CODE   = Path(os.environ.get("KNEE_CODE",   "/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline"))
DATA   = Path(os.environ.get("KNEE_DATA",   "/kaggle/input/competitions/rsna-knee-abnormality-detection"))
CACHE  = Path(os.environ.get("KNEE_CACHE",  "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache"))
LABELS = Path(os.environ.get("KNEE_LABELS", "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv"))
BASE   = Path(os.environ.get("KNEE_BASE",   "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv"))
for name, p in dict(CODE=CODE, DATA=DATA, CACHE=CACHE, LABELS=LABELS, BASE=BASE).items():
    assert p.exists(), f"{name} not found: {p}"
for f in ("geom.py", "tests/smoke_v6.py"):
    assert (CODE / f).exists(), f"{f} missing: dataset does not hold knee_pipeline_v6"
print("GPU0:", CNX_S, "| GPU1:", CNX_T, "at lr 2.5e-5 then 1e-4")

GPU0: convnext_small.in12k_ft_in1k | GPU1: convnext_tiny.in12k_ft_in1k at lr 2.5e-5 then 1e-4


In [2]:
# tests: if any fails, the assert stops the notebook BEFORE training starts
for t in ("tests/test_geom.py", "tests/smoke_v6.py"):
    r = subprocess.run([sys.executable, str(CODE / t)], capture_output=True, text=True, cwd=str(CODE))
    print(f"== {t} (exit {r.returncode})\n{r.stdout[-800:]}")
    assert r.returncode == 0, f"{t} failed:\n{r.stderr[-1500:]}"
EXTRA = ["--head", "tokens"]
print("extra train args:", EXTRA)

== tests/test_geom.py (exit 0)
canonical orientation identical for 24 acquisition conventions
right/left mirror normalisation OK for 3 planes
laterality rules OK
ALL GEOM TESTS PASS

== tests/smoke_v6.py (exit 0)
head=series: forward/backward OK, absent-series masking OK, params 5.69M
head=tokens: forward/backward OK, absent-series masking OK, params 5.18M
backbone convnext_tiny.in12k_ft_in1k: OK | vit_size None | params 28.9M
backbone efficientnet_b2: OK | vit_size None | params 8.9M
backbone vit_small_patch14_dinov2.lvd142m: OK | vit_size 168 | params 22.5M
backbone vit_small_patch16_224.dino: OK | vit_size 160 | params 22.6M
backbone deit_tiny_patch16_224.fb_in1k: OK | vit_size 160 | params 6.4M
v5-style state_dict loads into head='series'
KneeDS geometry parity OK
ALL V6 SMOKE TESTS PASS

extra train args: ['--head', 'tokens']


In [3]:
BASE_ARGS = ["--cache_dir", str(CACHE), "--labels", str(LABELS), "--epochs", "12", "--pretrained",
             "--workers", "2", "--train_slices", "16", "--bs", "4", "--accum", "2"]
PLAN = {0: [("v6cnxsmall_f0",   ["--backbone", CNX_S])],
        1: [("v6cnxt_lr25e6_f0", ["--backbone", CNX_T, "--lr_backbone", "2.5e-5"]),
            ("v6cnxt_lr1e4_f0",  ["--backbone", CNX_T, "--lr_backbone", "1e-4"])]}
NAMES = [n for jobs in PLAN.values() for n, _ in jobs]
FIRST_EPOCH_S, STALL_S = 30 * 60, 45 * 60
LOGS, RC = {}, {}
def n_epochs(f): return sum("ep " in l for l in f.read_text().splitlines()) if f.exists() else 0
def run_one(gpu, name, jx):
    out = W / name
    if (out / "oof.csv").exists():
        print(f"[{name}] oof.csv exists, skipped"); RC[name] = 0; return
    cmd = [sys.executable, str(CODE / "train.py")] + BASE_ARGS + EXTRA + jx + ["--only_folds", "0", "--out", str(out)]
    LOGS[name] = W / f"{name}.log"
    print(f"[{name}] GPU {gpu}: {' '.join(cmd)}", flush=True)
    fh = open(LOGS[name], "w")
    p = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)), cwd=str(CODE))
    t0 = last_t = time.time(); seen = 0
    while p.poll() is None:
        time.sleep(30)
        k = n_epochs(LOGS[name])
        if k > seen: seen, last_t = k, time.time()
        limit = FIRST_EPOCH_S if seen == 0 else STALL_S
        if time.time() - (t0 if seen == 0 else last_t) > limit:
            p.kill(); p.wait(); RC[name] = "KILLED by watchdog (no progress)"; print(f"[{name}] {RC[name]}", flush=True); fh.close(); return
    fh.close(); RC[name] = p.returncode
    print(f"[{name}] exit code {RC[name]}" + (" (killed by the OS: probably out of RAM)" if RC[name] == -9 else ""), flush=True)
def run_gpu(gpu, jobs):
    for name, jx in jobs: run_one(gpu, name, jx)
threads = [threading.Thread(target=run_gpu, args=(g, jobs)) for g, jobs in PLAN.items()]
for th in threads: th.start()
t0 = time.time()
while any(th.is_alive() for th in threads):
    time.sleep(600)
    last = {}
    for n in NAMES:
        f = LOGS.get(n)
        ls = [l for l in f.read_text().splitlines() if "ep " in l] if f is not None and f.exists() else []
        last[n] = ls[-1] if ls else "(not started or no epoch yet)"
    print(f"[{(time.time()-t0)/3600:.2f} h] {json.dumps(last, indent=1)}", flush=True)
for th in threads: th.join()
print("all jobs finished | results:", RC)
for n, rc in RC.items():
    if rc != 0:
        print(f"--- {n} FAILED, log tail ---"); print("\n".join(LOGS[n].read_text().splitlines()[-25:]))

[v6cnxsmall_f0] GPU 0: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --epochs 12 --pretrained --workers 2 --train_slices 16 --bs 4 --accum 2 --head tokens --backbone convnext_small.in12k_ft_in1k --only_folds 0 --out /kaggle/working/v6cnxsmall_f0
[v6cnxt_lr25e6_f0] GPU 1: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --epochs 12 --pretrained --workers 2 --train_slices 16 --bs 4 --accum 2 --head tokens --backbone convnext_tiny.in12k_ft_in1k --lr_backbone 2.5e-5 --only_folds 0 --out /kaggle/working/v6cnxt_lr25e6_f0
[0.17 h] {
 "v6cnxsmall_f0": "(not started or no epoch yet)",


In [4]:
# ---- fold-0 comparison on the FIXED old labels (confident: y>=0.9 or <=0.1) ----
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
LAB = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
KEY4 = LAB[:4]
base = pd.read_csv(BASE)
Y = {l: base[f"y_{l}"].values for l in LAB}
def auc_l(y, p):
    m = (y >= 0.9) | (y <= 0.1)
    return roc_auc_score((y[m] >= 0.9).astype(int), p[m]) if m.sum() > 20 and len({*(y[m] >= 0.9)}) == 2 else np.nan
PRED = {"baseline": base}
for name in NAMES:
    f = W / name / "oof.csv"
    if f.exists():
        d = pd.read_csv(f); assert len(d) == len(base), f"{name}: row count differs from baseline"
        PRED[name.replace("_f0", "")] = d
cx = [p for p in Path("/kaggle/input").rglob("oof_v6convnext.csv")]
if cx: PRED["convnext"] = pd.read_csv(cx[0]); print("convnext OOF found:", cx[0])
else: print("convnext OOF not attached: skipping the convnext comparison")
assert any(k not in ("baseline", "convnext") for k in PRED), "no DINO oof.csv found: the training jobs failed, read the log tails printed by the training cell"
ref = next(v for k, v in PRED.items() if k not in ("baseline", "convnext"))
mask = ref[f"p_{LAB[0]}"].notna().values            # fold-0 validation studies = the rows this screen predicted
print("fold-0 validation studies:", int(mask.sum()))
RK = {n: {l: rankdata(d.loc[mask, f"p_{l}"].values) / mask.sum() for l in LAB} for n, d in PRED.items()}
names = [n for n in RK if n not in ("baseline", "convnext")]
if "convnext" in RK:
    for n in names: RK[f"ens {n}+convnext"] = {l: (RK[n][l] + RK["convnext"][l]) / 2 for l in LAB}
T = pd.DataFrame({n: {l: auc_l(Y[l][mask], r[l]) for l in LAB} for n, r in RK.items()})
T.loc["MACRO"] = T.loc[LAB].mean(); T.loc["KEY-4"] = T.loc[KEY4].mean()
print("\nFold 0 only (ranks within fold):\n", T.round(3).to_string())
print("\nreference from the ConvNeXt run: fold-0 raw macro 0.8328, baseline 0.7795. Seed noise ~0.01.")
print("rule: an arm is worth a 5-fold only if it beats ConvNeXt-T (lr 5e-5) fold 0 = 0.833 by more than ~0.015 macro / KEY-4. For ConvNeXt-Small also check the epoch time in its log: a 5-fold costs about (epoch seconds x 12 x 3 folds) per GPU.")

convnext OOF found: /kaggle/input/notebooks/theokorir/knee-v6-convnext-5fold/oof_v6convnext.csv
fold-0 validation studies: 882

Fold 0 only (ranks within fold):
                   baseline  v6cnxsmall  v6cnxt_lr25e6  v6cnxt_lr1e4  convnext  ens v6cnxsmall+convnext  ens v6cnxt_lr25e6+convnext  ens v6cnxt_lr1e4+convnext
ACL                  0.833       0.880          0.893         0.877     0.892                    0.891                       0.898                      0.891
MCL                  0.713       0.778          0.790         0.757     0.805                    0.797                       0.801                      0.788
Medial Meniscus      0.757       0.875          0.867         0.863     0.870                    0.877                       0.871                      0.870
Lateral Meniscus     0.715       0.754          0.763         0.738     0.776                    0.769                       0.771                      0.761
Medial OA            0.861       0.876          